# QLoRA fine-tune: Qwen2.5 1.5B transaction categoriser

Run on **Kaggle** with *Accelerator: GPU T4 x2* (one GPU is used) and *Internet: on*.
The repo is cloned fresh, the synthetic data is regenerated (deterministic), the model is
fine-tuned, then the 217 hand-labelled test rows are predicted and scored with the same
script as the TF-IDF and Gemini baselines.

Change the settings in the next cell between runs (rank, learning rate, epochs) and note
each run's scores in the results table.

In [ ]:
# Run settings: change these between experiments
RUN_NAME = "r16_lr2e-4_ep1"
LORA_RANK = 16
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQ_LEN = 256   # narrations plus the JSON answer fit well inside this
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"


In [ ]:
!git clone -q https://github.com/rc-mathew/upi-transaction-categoriser-qlora repo
%cd repo
!pip install -q unsloth faker
!python scripts/generate_synthetic.py --n 24000 --out data/synthetic/train.csv
!python scripts/build_sft_data.py data/synthetic/train.csv data/synthetic


## Load the 4-bit base model and attach LoRA adapters

In [ ]:
import sys, json, time, csv, os
sys.path.insert(0, "src")
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)

# General-ability check, before: the same prompts are asked again after training
GENERAL = ["What is the capital of Australia?", "Write one sentence explaining what a credit score is.", "What is 17 * 23?"]
def ask(prompts, max_new_tokens=60):
    FastLanguageModel.for_inference(model)
    outs = []
    for p in prompts:
        ids = tokenizer.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, return_tensors="pt").to("cuda")
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
        outs.append(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return outs
before = ask(GENERAL)
for p, a in zip(GENERAL, before): print(f"Q: {p}\nA: {a}\n")

model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)
model.print_trainable_parameters()


## Train (loss only on the assistant's JSON, not the prompt)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

ds = load_dataset("json", data_files={"train": "data/synthetic/sft_train.jsonl", "val": "data/synthetic/sft_val.jsonl"})
ds = ds.map(lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in b["messages"]]}, batched=True, remove_columns=["messages"])

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_ratio=0.03,
        logging_steps=25, eval_strategy="steps", eval_steps=100,
        optim="adamw_8bit", weight_decay=0.01, seed=0, output_dir="outputs", report_to="none"))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

t0 = time.time()
stats = trainer.train()
print(f"trained in {(time.time() - t0) / 60:.1f} min")


Check the log above: validation loss should fall and level off. If it rises while training loss keeps falling, the model is overfitting: train fewer steps or lower the learning rate.

## Predict the hand-labelled test set

In [ ]:
from categoriser.prompt import chat
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
test = list(csv.DictReader(open("data/test/test.csv", encoding="utf-8")))

rows, BS = [], 16
for i in range(0, len(test), BS):
    batch = test[i:i + BS]
    texts = [tokenizer.apply_chat_template(chat(r, with_answer=False), tokenize=False, add_generation_prompt=True) for r in batch]
    enc = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
    t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=64, do_sample=False)
    ms = (time.perf_counter() - t0) * 1000 / len(batch)  # GPU number; the CPU latency that matters is measured in week 11
    for r, o in zip(batch, out):
        rows.append([r["narration"], tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip(), round(ms, 1)])

os.makedirs("runs", exist_ok=True)
with open(f"runs/qwen_{RUN_NAME}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["narration", "output", "latency_ms"]); w.writerows(rows)
print(rows[0])


In [ ]:
!cd src && python -m categoriser.evaluate ../data/test/test.csv ../runs/qwen_{RUN_NAME}.csv


## General-ability check, after

In [ ]:
after = ask(GENERAL)
for p, b, a in zip(GENERAL, before, after): print(f"Q: {p}\nbefore: {b}\nafter:  {a}\n")


## Save the adapter and predictions (download them from the Output panel)

In [ ]:
model.save_pretrained(f"/kaggle/working/adapter_{RUN_NAME}")
tokenizer.save_pretrained(f"/kaggle/working/adapter_{RUN_NAME}")
!cp runs/qwen_{RUN_NAME}.csv /kaggle/working/
!ls -la /kaggle/working
